# Train Transformer-Summarizer on a Colab GPU

Before running anything: **Runtime > Change runtime type > T4 GPU > Save.**

Then **Runtime > Run all**. This notebook clones the project from GitHub, installs dependencies, downloads CNN/DailyMail, trains the model on GPU, runs evaluation, and packages the results for you to download.

Free Colab GPU sessions can disconnect after a period of inactivity or after ~12 hours. If you expect a long run, turn on the optional Google Drive mount below so checkpoints survive a disconnect — otherwise just re-run the final "package results" cell any time to grab whatever has been produced so far.

In [ ]:
!nvidia-smi

## Optional: mount Google Drive for crash-safe checkpoints

Set `USE_DRIVE = True` below to save checkpoints/logs directly to your Drive instead of the ephemeral Colab disk. You'll be prompted to authorize access — that prompt and the authorization itself are between you and Google; nothing here handles your credentials.

In [ ]:
USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    CHECKPOINT_DIR = "/content/drive/MyDrive/Transformer-Summarizer/checkpoints"
    LOG_DIR = "/content/drive/MyDrive/Transformer-Summarizer/logs"
else:
    CHECKPOINT_DIR = "checkpoints"
    LOG_DIR = "logs"

print("checkpoint_dir:", CHECKPOINT_DIR)
print("log_dir:", LOG_DIR)

## Clone the repo and install dependencies

In [ ]:
import os

REPO_URL = "https://github.com/kshitizxzz/Transformer-Summarizer.git"
REPO_DIR = "Transformer-Summarizer"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
%cd {REPO_DIR}
!git pull

In [ ]:
!pip install -q -r requirements.txt

## Download the dataset

A GPU can chew through far more data than a laptop CPU, so the caps here are bigger than the CPU defaults. Set any of these to `0` to use the full split (full train is ~287k examples).

In [ ]:
MAX_TRAIN = 50000
MAX_VAL = 3000
MAX_TEST = 3000

!python scripts/download_dataset.py --max_train {MAX_TRAIN} --max_val {MAX_VAL} --max_test {MAX_TEST}

## Train

`--device cuda` is required — the trainer does not auto-detect GPU vs CPU beyond `cuda`/`cpu`, so leaving this off would still work (it auto-picks `cuda` when available) but being explicit avoids surprises. Adjust `EPOCHS`/`BATCH_SIZE` as needed; check `python -m src.training.train --help` for every other hyperparameter (`--d_model`, `--num_layers`, etc.).

In [ ]:
EPOCHS = 10
BATCH_SIZE = 64

!python -m src.training.train \
    --train_path data/train.csv \
    --val_path data/val.csv \
    --device cuda \
    --epochs {EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --checkpoint_dir {CHECKPOINT_DIR} \
    --log_dir {LOG_DIR}

## Evaluate

Computes perplexity over the full test split plus ROUGE + qualitative examples over a capped subset, and writes `logs/eval_results.json` (or the Drive path, if `USE_DRIVE` is on) — this is the file the Streamlit **Results** page reads.

In [ ]:
!python -m src.evaluation.evaluate \
    --data_path data/test.csv \
    --checkpoint {CHECKPOINT_DIR}/best.pt \
    --vocab_path data/vocab.json \
    --device cuda \
    --max_eval_examples 200 \
    --output_path {LOG_DIR}/eval_results.json

## Get your results

If `USE_DRIVE` is on, everything is already saved to your Drive under `MyDrive/Transformer-Summarizer/`. Otherwise, run this cell to zip up the checkpoints, training history, eval results, and vocab, and download them through the browser.

In [ ]:
if not USE_DRIVE:
    !zip -rq outputs.zip {CHECKPOINT_DIR} {LOG_DIR} data/vocab.json
    from google.colab import files
    files.download('outputs.zip')
else:
    print("USE_DRIVE is on — results are already in your Google Drive, nothing to download here.")

## Bring the results back to your local project

Unzip `outputs.zip` (or copy from Drive) so that `checkpoints/`, `logs/`, and `data/vocab.json` land at the same paths inside your local `Transformer-Summarizer/` folder, overwriting the empty placeholders. Then locally:

```bash
streamlit run ui/Home.py
```

The Summarize, Training Analytics, and Results pages will now have a real trained model and real metrics to show. These files are intentionally excluded from git (see `.gitignore`), so there's nothing to commit — just drop them in and run the app.